In [1]:
import random

# --- CONFIGURATION & SAMPLE DATA ---
POPULATION_SIZE = 100
MAX_GENERATIONS = 500
MUTATION_RATE = 0.05
PERFECT_SCORE = 1000

# Fixed university data
CLASSES = ["CS101", "BIO202", "CHEM301", "MATH101", "ENG102", "PHY201"]
PROFESSORS = ["prof_alan", "prof_sarah", "prof_mike", "prof_dave", "prof_alan", "prof_sarah"]
ROOMS = ["room_101", "room_102", "auditorium_a"]
ROOM_CAPACITIES = {"room_101": 40, "room_102": 40, "auditorium_a": 150}
CLASS_SIZES = {"CS101": 35, "BIO202": 38, "CHEM301": 120, "MATH101": 30, "ENG102": 25, "PHY201": 80}
TIMES = ["mon_09am", "mon_11am", "tue_09am", "tue_11am", "fri_06pm"]
TOTAL_CLASSES = len(CLASSES)

# Helper class representing a single scheduled lecture
class Lecture:
    def __init__(self, class_id, prof, room, time):
        self.class_id = class_id
        self.prof = prof
        self.room = room
        self.time = time

# --- GENETIC ALGORITHM CORE FUNCTIONS ---

def random_schedule():
    # Initializes a completely random timetable configuration
    schedule = []
    for i in range(TOTAL_CLASSES):
        lecture = Lecture(
            class_id=CLASSES[i],
            prof=PROFESSORS[i],
            room=random.choice(ROOMS),
            time=random.choice(TIMES)
        )
        schedule.append(lecture)
    return schedule

def evaluate_fitness(schedule):
    score = 1000

    # Track occupations to identify real-world collisions
    room_time_registry = set()
    prof_time_registry = set()

    for c in schedule:
        # Check hard constraints: Double booking
        room_key = (c.room, c.time)
        prof_key = (c.prof, c.time)

        if room_key in room_time_registry or prof_key in prof_time_registry:
            score -= 100

        room_time_registry.add(room_key)
        prof_time_registry.add(prof_key)

        # Check hard constraints: Room capacity overload
        if CLASS_SIZES[c.class_id] > ROOM_CAPACITIES[c.room]:
            score -= 50

        # Check soft constraints: Preferences
        if c.time == "fri_06pm":
            score -= 5

    return score

def select_parent(pop, scores):
    # Simple tournament selection to pick a strong candidate parent
    k = 3
    selected = random.sample(list(zip(pop, scores)), k)
    selected.sort(key=lambda x: x[1], reverse=True)
    return selected[0][0]

def get_top_performers(pop, scores, count):
    # Sorts the current population by fitness and grabs the elite top items
    paired = list(zip(pop, scores))
    paired.sort(key=lambda x: x[1], reverse=True)
    return [item[0] for item in paired[:count]]

def crossover(pa, pb):
    mid = TOTAL_CLASSES // 2
    # Slices half of the lecture assignments from parent A and half from parent B
    child_classes = pa[:mid] + pb[mid:]

    # Deep copy objects so they don't share memory space variables
    child = [Lecture(c.class_id, c.prof, c.room, c.time) for c in child_classes]
    return child

def mutate(schedule):
    # Randomly picks one course slot and modifies its physical location or timing
    c = random.choice(schedule)
    c.room = random.choice(ROOMS)
    c.time = random.choice(TIMES)
    return schedule

# --- EVOLUTIONARY ENGINE MAIN CONTROL ---

def start_evolution():
    pop = [random_schedule() for _ in range(POPULATION_SIZE)]
    print("Beginning evolutionary algorithm scheduling process...\n")

    for gen in range(1, MAX_GENERATIONS + 1):
        scores = [evaluate_fitness(s) for s in pop]
        max_score = max(scores)
        best_index = scores.index(max_score)

        # Log generation milestones or early success metrics
        if gen == 1 or gen % 50 == 0 or max_score == PERFECT_SCORE:
            print(f"Generation {gen} | Max Fitness Score: {max_score}/{PERFECT_SCORE}")

        if max_score == PERFECT_SCORE:
            return pop[best_index], gen

        next_pop = get_top_performers(pop, scores, 5)

        while len(next_pop) < POPULATION_SIZE:
            pa = select_parent(pop, scores)
            pb = select_parent(pop, scores)
            child = crossover(pa, pb)

            if random.random() < MUTATION_RATE:
                child = mutate(child)

            next_pop.append(child)

        pop = next_pop

    # Fallback to display the best overall found if generation loop times out
    scores = [evaluate_fitness(s) for s in pop]
    best_index = scores.index(max(scores))
    return pop[best_index], MAX_GENERATIONS

# --- RUN EXECUTION & PRINT THE RESULTS ---
best_schedule, generations_taken = start_evolution()

print(f"\nOptimization Finished! Final configuration output after {generations_taken} generations:\n")
print(f"{'Course':<10} | {'Professor':<12} | {'Assigned Room':<15} | {'Time Slot':<10}")
print("-" * 60)
for lect in best_schedule:
    print(f"{lect.class_id:<10} | {lect.prof:<12} | {lect.room:<15} | {lect.time:<10}")


Beginning evolutionary algorithm scheduling process...

Generation 1 | Max Fitness Score: 995/1000
Generation 3 | Max Fitness Score: 1000/1000

Optimization Finished! Final configuration output after 3 generations:

Course     | Professor    | Assigned Room   | Time Slot 
------------------------------------------------------------
CS101      | prof_alan    | auditorium_a    | mon_09am  
BIO202     | prof_sarah   | room_102        | tue_11am  
CHEM301    | prof_mike    | auditorium_a    | tue_09am  
MATH101    | prof_dave    | room_101        | mon_11am  
ENG102     | prof_alan    | room_102        | tue_09am  
PHY201     | prof_sarah   | auditorium_a    | mon_11am  
